# SONIC-Lite G1: Colab train → test

Run the cells from top to bottom. The extraction cell supports a small 130-clip setup or the complete G1 archive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/YigitGunduc/robot.git'
REPO_DIR = Path('/content/robot')
DATASET_ROOT = Path('/content/drive/MyDrive/Datasets/bones-seed')
OUTPUT_DIR = Path('/content/drive/MyDrive/sonic-lite-results')

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO_DIR)
print('Dataset:', DATASET_ROOT)
print('Extracted CSVs:', DATASET_ROOT / 'g1' / 'csv')
print('Results:', OUTPUT_DIR)

In [ ]:
# Compatibility patch for the mjlab version installed in Colab.
# Some releases do not expose history_ordering on ObservationGroupCfg.
env_cfg_file = REPO_DIR / 'sonic_lite_g1' / 'env_cfg.py'
env_cfg_text = env_cfg_file.read_text()
env_cfg_text = env_cfg_text.replace('        history_ordering="time",\n', '')
env_cfg_file.write_text(env_cfg_text)
print('mjlab observation-config compatibility applied')

In [ ]:
# Extract BONES-SEED. Use 'needed' for the first 30 stand + 100 walk clips.
# Change to 'all' only if Google Drive has enough free space for the full archive.
import shutil
import tarfile

EXTRACT_MODE = 'needed'  # 'needed' or 'all'
ARCHIVE = DATASET_ROOT / 'g1.tar.gz'
CSV_ROOT = DATASET_ROOT / 'g1' / 'csv'

if not ARCHIVE.exists():
    raise FileNotFoundError(f'Archive not found: {ARCHIVE}')

NEEDED_MARKER = DATASET_ROOT / '.sonic_lite_clean_130_clips_ready'
FULL_MARKER = DATASET_ROOT / '.g1_full_extracted'

if EXTRACT_MODE == 'all':
    if FULL_MARKER.exists() and CSV_ROOT.exists():
        print('Full G1 dataset already extracted; skipping extraction.')
    else:
        print('Extracting the complete G1 archive...')
        subprocess.run(['tar', '-xzf', str(ARCHIVE), '-C', str(DATASET_ROOT)], check=True)
        FULL_MARKER.write_text('Full G1 archive extracted\n')
        print('Full extraction complete.')
else:
    import sys
    sys.path.insert(0, str(REPO_DIR))
    from sonic_lite_g1.data.select_bones import classify

    wanted = {'stand': 30, 'walk': 100}
    found = {'stand': 0, 'walk': 0}
    CSV_ROOT.mkdir(parents=True, exist_ok=True)

    # Reuse files already saved in Drive, even if the marker was not created.
    for existing in CSV_ROOT.rglob('*.csv'):
        group = classify(str(existing))
        if group in found and found[group] < wanted[group]:
            found[group] += 1

    if NEEDED_MARKER.exists() or all(found[key] >= wanted[key] for key in wanted):
        NEEDED_MARKER.write_text('30 clean stand clips + 100 clean walk clips extracted\n')
        print('Needed BONES clips already extracted; skipping archive scan.')
    else:
        print('Extracting only clean clips:', wanted, 'already found:', found)
        with tarfile.open(ARCHIVE, mode='r|gz') as archive:
            for member in archive:
                name = member.name.lstrip('/')
                if not name.endswith('.csv') or '/g1/csv/' not in f'/{name}':
                    continue
                group = classify(name)
                if group not in wanted or found[group] >= wanted[group]:
                    continue
                destination = DATASET_ROOT / name
                if destination.exists():
                    found[group] += 1
                    continue
                destination.parent.mkdir(parents=True, exist_ok=True)
                source = archive.extractfile(member)
                if source is None:
                    continue
                with source, destination.open('wb') as target:
                    shutil.copyfileobj(source, target)
                found[group] += 1
                print(group, found[group], '/', wanted[group], destination.name)
                if all(found[key] >= wanted[key] for key in wanted):
                    break
        if not all(found[key] >= wanted[key] for key in wanted):
            raise RuntimeError(f'Could not extract all needed clips: {found}')
        NEEDED_MARKER.write_text('30 clean stand clips + 100 clean walk clips extracted\n')
        print('Needed extraction complete and persisted in Drive.')

print('CSV root exists:', CSV_ROOT.exists())
print('CSV count:', len(list(CSV_ROOT.rglob('*.csv'))) if CSV_ROOT.exists() else 0)

In [ ]:
# Compatibility patch for mjlab releases whose ObservationGroupCfg does not
# expose history_ordering. This leaves the model and training logic unchanged.
from pathlib import Path

env_cfg_file = REPO_DIR / 'sonic_lite_g1' / 'env_cfg.py'
env_cfg_text = env_cfg_file.read_text()
env_cfg_text = env_cfg_text.replace('        history_ordering="time",\n', '')
env_cfg_file.write_text(env_cfg_text)
print('mjlab observation-config compatibility applied')

In [ ]:
# Install the repository and its mjlab/RSL-RL dependencies.
%pip install -q -e .

## Train

Defaults are the initial easy curriculum: 30 standing clips and 100 walking clips, 1,024 environments, and 5,000 PPO iterations.

In [ ]:
import os
import subprocess

os.environ['MUJOCO_GL'] = 'egl'
os.environ['WANDB_MODE'] = 'offline'

command = [
    'python', '-u', 'scripts/train_and_evaluate.py',
    '--dataset-root', str(DATASET_ROOT),
    '--work-dir', str(OUTPUT_DIR),
    '--max-stand', '30',
    '--max-walk', '100',
    '--max-turn', '0',
    '--max-crouch', '0',
    '--max-jog', '0',
    '--num-envs', '256',
    '--max-iterations', '5000',
    '--video-length', '600',
]
print('Running:', ' '.join(command))
process = subprocess.Popen(
    command, cwd=str(REPO_DIR), env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
print('RETURN CODE (train_and_evaluate.py):', return_code)
if return_code != 0:
    raise RuntimeError(f'train_and_evaluate.py failed with return code {return_code}')

## Test and inspect results

The training script evaluates the newest checkpoint with one environment, records an MP4, and writes scalar training summaries.

In [ ]:
import json
from IPython.display import Video, display

metrics_file = OUTPUT_DIR / 'training_metrics.json'
video_file = OUTPUT_DIR / 'trained_action.mp4'

if metrics_file.exists():
    metrics = json.loads(metrics_file.read_text())
    print('Metric tags:', len(metrics))
    for tag, values in list(metrics.items())[:20]:
        print(tag, 'first=', values['first'], 'last=', values['last'], 'min=', values['min'], 'max=', values['max'])
else:
    print('Metrics file not found:', metrics_file)

if video_file.exists():
    display(Video(str(video_file), embed=True))
else:
    print('Video not found:', video_file)